# 02 — Feature Extraction

**Project:** The Geometry of Collective Attention  
**Author:** Khadidiatou Cissé  
**Date:** April 2026  
**Input:** `data/processed/videos_t0.csv`  
**Output:** `data/processed/features.csv`

---

## What this notebook does

Extracts four groups of content-side features for every video:

| Group | Type | Source | Speed |
|---|---|---|---|
| A — Audio | pitch, energy, tempo, ZCR | yt-dlp + librosa | slow (~30s/video) |
| B — Visual | colour, brightness, complexity | thumbnail URL + Pillow | fast (~0.5s/video) |
| C — Text | sentiment, readability, structure | title + description | fast (~0.1s/video) |
| D — Structural | duration, pacing, engagement density | API fields | instant |

**Recommended first run:** disable audio download (`DOWNLOAD_AUDIO = False`).  
This completes Groups B, C, D in a few minutes and lets you verify the pipeline.  
Enable audio later in a dedicated long run.

---

## Mathematical context

These features will be used in three roles:

1. **Baseline models (M1, M2):** direct predictors of scalar engagement rates  
2. **Sheaf coherence (§4):** inputs to the modality synchronisation maps —  
   we will compute cross-modal correlations (e.g. does audio energy match  
   visual brightness? does speech sentiment match title sentiment?)  
3. **Curvature interpretation:** once we have engagement trajectories in  
   notebook 03, we will ask: which of these features predicts *where* the  
   curvature peaks occur in the video timeline?

The last question is the most novel. A curvature peak at $t^* = 0.6T$  
(60% into the video) is an *event* — and the content features tell us  
what kind of content tends to produce events at that position.

In [ ]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path('..').resolve()))
from src.features import (
    extract_all_features,
    extract_text_features,
    extract_visual_features,
    extract_structural_features,
    download_thumbnail,
)

logging.basicConfig(level=logging.WARNING)  # suppress info logs in notebook

# ── Configure here ─────────────────────────────────────────────────────────
DOWNLOAD_AUDIO = False   # Set True for full pipeline (takes ~1hr for 800 videos)
AUDIO_DIR      = Path('../data/raw/audio')
THUMB_DIR      = Path('../data/raw/thumbnails')
# ───────────────────────────────────────────────────────────────────────────

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
THUMB_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')
print(f'Audio download: {DOWNLOAD_AUDIO}')

## 1. Load the collection output

In [ ]:
df = pd.read_csv('../data/processed/videos_t0.csv')
print(f'Loaded {len(df)} videos, {len(df.columns)} columns.')
print(f'Categories: {df["our_category_label"].value_counts().to_dict()}')
df.head(2)

## 2. Test on one video first

Always test on a single row before running the full pipeline.  
This catches import errors, missing libraries, or network issues early.

In [ ]:
test_row = df.iloc[0]
print(f'Testing on: "{test_row["title"]}"')
print(f'Category : {test_row["our_category_label"]}')
print()

test_features = extract_all_features(
    row=test_row,
    audio_dir=AUDIO_DIR,
    thumb_dir=THUMB_DIR,
    download_audio_flag=DOWNLOAD_AUDIO,
)

print(f'Extracted {len(test_features)} features.')
print()
for k, v in sorted(test_features.items()):
    if k != 'video_id':
        print(f'  {k:<35} = {v}')

## 3. Run on all videos

The progress bar shows estimated time remaining.  
If a video fails, `extract_all_features` returns NaN for that group  
rather than crashing — the pipeline is fault-tolerant.

In [ ]:
all_features = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting features'):
    feats = extract_all_features(
        row=row,
        audio_dir=AUDIO_DIR,
        thumb_dir=THUMB_DIR,
        download_audio_flag=DOWNLOAD_AUDIO,
    )
    all_features.append(feats)

df_features = pd.DataFrame(all_features)
print(f'\nExtracted {len(df_features.columns)} features for {len(df_features)} videos.')

## 4. Merge with the engagement data

In [ ]:
# Join on video_id
engagement_cols = [
    'video_id', 'our_category', 'our_category_label',
    'view_count', 'like_count', 'comment_count',
    'like_rate', 'comment_rate', 'engagement_rate',
    'days_since_publication', 'views_per_day',
]
engagement_cols = [c for c in engagement_cols if c in df.columns]

df_merged = df[engagement_cols].merge(df_features, on='video_id', how='left')

print(f'Merged dataset: {df_merged.shape[0]} rows × {df_merged.shape[1]} columns')

# Check for missing values by feature group
group_prefixes = {'audio': 'a_', 'visual': 'v_', 'text': 't_', 'structural': 'd_'}
print('\nMissing values by feature group:')
for gname, prefix in group_prefixes.items():
    cols = [c for c in df_merged.columns if c.startswith(prefix)]
    if cols:
        pct_missing = df_merged[cols].isnull().mean().mean() * 100
        print(f'  {gname:<12} ({len(cols)} features): {pct_missing:.1f}% missing')

## 5. Feature visualisations

### 5.1 Text sentiment by category

Title sentiment varies strongly across categories —  
comedy and wellness titles should be positive, political titles negative or polarised.

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0d0f14', 'axes.facecolor': '#131620',
    'axes.edgecolor': '#2a3045', 'text.color': '#c8bfa8',
    'axes.labelcolor': '#c8bfa8', 'xtick.color': '#7a8099',
    'ytick.color': '#7a8099', 'grid.color': '#1e2332',
    'grid.linewidth': 0.6, 'font.family': 'monospace', 'font.size': 10,
})

CAT_ORDER = [
    'Science & Education', 'Mathematics & Philosophy', 'Meditation & Wellness',
    'Cooking', 'Beauty & Fashion', 'Personal Vlog',
    'Finance & Investing', 'Gaming', 'Comedy', 'Political Commentary'
]
PALETTE = [
    '#3ecfb8', '#9b7fe8', '#5aa8f0', '#e8a832', '#e05c4a',
    '#78c878', '#c9a84c', '#f0ead8', '#e87850', '#d4635a'
]
cat_colour = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(CAT_ORDER)}

cats_present = df_merged['our_category_label'].unique()

fig, ax = plt.subplots(figsize=(12, 5))

groups = [
    df_merged[df_merged['our_category_label'] == cat]['t_title_sentiment'].dropna()
    for cat in CAT_ORDER if cat in cats_present
]
labels_present = [cat for cat in CAT_ORDER if cat in cats_present]
cols_present   = [cat_colour.get(c, '#7a8099') for c in labels_present]

if groups:
    bp = ax.boxplot(
        groups, patch_artist=True,
        medianprops=dict(color='#f0ead8', linewidth=1.8),
        whiskerprops=dict(color='#7a8099'),
        capprops=dict(color='#7a8099'),
        flierprops=dict(marker='o', markersize=2.5,
                        markerfacecolor='#7a8099', linestyle='none'),
    )
    for patch, col in zip(bp['boxes'], cols_present):
        patch.set_facecolor(col)
        patch.set_alpha(0.45)

    ax.axhline(0, color='#e8a832', linewidth=1, linestyle='--', alpha=0.5,
               label='neutral')
    ax.set_xticks(range(1, len(labels_present) + 1))
    ax.set_xticklabels(
        [l.replace(' & ', '\n& ') for l in labels_present], fontsize=7.5
    )
    ax.set_ylabel('VADER compound sentiment', fontsize=9)
    ax.set_title('Title sentiment by content category', fontsize=11, color='#e8e4dc')
    ax.grid(True, axis='y')
    ax.legend()

plt.tight_layout()
Path('../results/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('../results/figures/02_title_sentiment.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

### 5.2 Visual complexity vs brightness

Each dot is a video, coloured by category.  
We expect beauty/gaming videos to cluster in high-brightness + high-complexity,  
meditation/wellness in low-complexity + moderate brightness.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for cat in CAT_ORDER:
    if cat not in cats_present:
        continue
    sub = df_merged[df_merged['our_category_label'] == cat]
    ax.scatter(
        sub['v_visual_complexity'],
        sub['v_brightness_mean'],
        c=cat_colour.get(cat, '#7a8099'),
        alpha=0.55, s=30, label=cat, edgecolors='none'
    )

ax.set_xlabel('Visual complexity (thumbnail pixel variance)', fontsize=9)
ax.set_ylabel('Brightness (mean luminosity)', fontsize=9)
ax.set_title('Thumbnail visual space — complexity vs brightness', fontsize=11,
             color='#e8e4dc')
ax.legend(fontsize=7, ncol=2, framealpha=0.2)
ax.grid(True)

plt.tight_layout()
fig.savefig('../results/figures/02_visual_scatter.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

### 5.3 Feature correlation heatmap

Which content features are correlated with engagement?  
And which content features are correlated with each other  
(collinearity — a modelling concern)?

In [ ]:
# Select numeric features with low missing rates
target_cols  = ['like_rate', 'comment_rate', 'engagement_rate']
feature_cols = [
    c for c in df_merged.columns
    if c.startswith(('t_', 'v_', 'd_'))
    and df_merged[c].dtype in [float, int, 'float64', 'int64']
    and df_merged[c].isnull().mean() < 0.3
]

if feature_cols:
    corr_cols = target_cols + feature_cols
    corr = df_merged[corr_cols].corr()

    fig, ax = plt.subplots(
        figsize=(max(8, len(corr_cols) * 0.45),
                 max(6, len(corr_cols) * 0.4))
    )

    im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=90, fontsize=6)
    ax.set_yticklabels(corr.index, fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.02)
    ax.set_title('Feature correlation matrix', fontsize=11, color='#e8e4dc')

    # Highlight target rows
    for i, col in enumerate(corr.columns):
        if col in target_cols:
            ax.add_patch(plt.Rectangle((i - 0.5, -0.5), 1, len(corr), 
                                        fill=False, edgecolor='#e8a832',
                                        lw=1.5))

    plt.tight_layout()
    fig.savefig('../results/figures/02_feature_correlation.png',
                dpi=150, bbox_inches='tight', facecolor='#0d0f14')
    plt.show()
else:
    print('No numeric features available yet. Run with audio for full matrix.')

### 5.4 Top features correlated with like_rate

Horizontal bar chart of absolute correlation with `like_rate`.  
This gives the first answer to: **which content features matter most?**

In [ ]:
if feature_cols:
    corr_with_like = (
        df_merged[feature_cols + ['like_rate']]
        .corr()['like_rate']
        .drop('like_rate')
        .abs()
        .sort_values(ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.barh(
        range(len(corr_with_like)),
        corr_with_like.values,
        color='#3ecfb8', alpha=0.7, edgecolor='none'
    )
    ax.set_yticks(range(len(corr_with_like)))
    ax.set_yticklabels(corr_with_like.index, fontsize=8)
    ax.set_xlabel('|Pearson correlation| with like_rate', fontsize=9)
    ax.set_title('Top 20 features by correlation with like_rate',
                 fontsize=11, color='#e8e4dc')
    ax.grid(True, axis='x')
    ax.invert_yaxis()

    plt.tight_layout()
    fig.savefig('../results/figures/02_top_features.png',
                dpi=150, bbox_inches='tight', facecolor='#0d0f14')
    plt.show()

    print('Top 5 features by correlation with like_rate:')
    print(corr_with_like.head(5).to_string())
else:
    print('No features to rank yet.')

## 6. Save the feature table

In [ ]:
out_path = Path('../data/processed/features.csv')
df_merged.to_csv(out_path, index=False)

print(f'Saved {df_merged.shape[0]} rows x {df_merged.shape[1]} columns -> {out_path}')

# Summary for the record
n_audio    = len([c for c in df_merged.columns if c.startswith('a_')])
n_visual   = len([c for c in df_merged.columns if c.startswith('v_')])
n_text     = len([c for c in df_merged.columns if c.startswith('t_')])
n_struct   = len([c for c in df_merged.columns if c.startswith('d_')])

print(f'\nFeature counts:')
print(f'  Audio      : {n_audio}')
print(f'  Visual     : {n_visual}')
print(f'  Text       : {n_text}')
print(f'  Structural : {n_struct}')
print(f'  Total      : {n_audio + n_visual + n_text + n_struct}')

## 7. What is still missing — and why it matters

This notebook produces the content-side features.  
What is intentionally absent:

**The engagement trajectory $\mathbf{e}(t)$** is not yet here because it  
requires re-querying the same videos at $t_1, t_2, t_3$ — done in notebook 03.  

**Audio features** require `DOWNLOAD_AUDIO = True` — run this as a  
separate long job, ideally overnight:

```bash
jupyter nbconvert --to notebook --execute notebooks/02_feature_extraction.ipynb \
    --ExecutePreprocessor.timeout=7200
```

**The key question this notebook raises:**  
Look at the top-5 features correlated with `like_rate`.  
If a *structural* feature (e.g. `d_log_duration`) ranks above any  
*content* feature, it means the baseline model (M1) is already capturing  
most of the variance, and the richer features add less than expected.  
This would motivate a stronger focus on the trajectory geometry as the  
genuinely new explanatory layer — exactly the motivation for models M3 and M4.